# Phase 1: Data Preparation and Filtering
This notebook filters UsnJrnl and LogFile datasets using Oh et al.'s methodology to reduce dataset size by 60-70% without compromising detection capability.

**Input Files**
* UsnJrnl (enriched with MFT timestamps): data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched.csv
* LogFile: data/validation/logfile/LoneWolf-LogFile.csv

**Output Files**
* Filtered UsnJrnl: data/validation/processed/phase1/LoneWolf_UsnJrnl_filtered.csv
* Filtered LogFile: data/validation/processed/phase1/LoneWolf_LogFile_filtered.csv

**Process**
UsnJrnl: 3-stage aggressive filtering (keep BDP events, first creation per file, delete/rename at monitored paths)
LogFile: Keep timestamp change events and file creation/deletion events

In [83]:
# Cell 1: Import libraries and set paths

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
USNJRNL_ENRICHED_PATH = BASE_DIR / 'data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched(filtered).csv'
LOGFILE_PATH = BASE_DIR / 'data/validation/logfile/LoneWolf-LogFile.csv'

# Output paths
OUTPUT_DIR = BASE_DIR / 'notebooks/LW Testing Pipeline/Outputs/trial 2/v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USNJRNL_FILTERED_PATH = OUTPUT_DIR / 'LoneWolf_UsnJrnl_filtered.csv'
LOGFILE_FILTERED_PATH = OUTPUT_DIR / 'LoneWolf_LogFile_filtered.csv'

print(f"UsnJrnl input: {USNJRNL_ENRICHED_PATH}")
print(f"LogFile input: {LOGFILE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


UsnJrnl input: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched(filtered).csv
LogFile input: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/logfile/LoneWolf-LogFile.csv
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1


# Part 1: UsnJrnl Aggressive Filtering

Strategy: 3-stage filtering to keep only detection-relevant events

* Stage 1: All Basic_Info_Changed events (potential timestomping)
* Stage 2: First File_Created event per FileReferenceNumber
* Stage 3: Delete/Rename events at monitored paths only (for tunneling detection)

Expected reduction: 60-70% of events

In [84]:
# Cell 2: Load enriched UsnJrnl

print("Loading enriched UsnJrnl...")
usnjrnl = pd.read_csv(USNJRNL_ENRICHED_PATH, low_memory=False)

# Drop the column (Empty)
usnjrnl = usnjrnl.drop(columns=["Carving Flag"])

print(f"Total events loaded: {len(usnjrnl):,}")
print(f"Columns: {list(usnjrnl.columns)}")
print(f"\nFirst 3 rows:")
print(usnjrnl.head(3))

Loading enriched UsnJrnl...
Total events loaded: 101,023
Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'FileReferenceNumber', 'FileName_MFT', 'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted', 'EventTime_Formatted', 'MFT_RecordNumber']

First 3 rows:
        TimeStamp(UTC+8)        USN File/Directory Name  \
0  04/01/18 16:56:07:567  209715200  f498ac39e16a30c8_1   
1  04/01/18 16:56:07:567  209715296  f498ac39e16a30c8_1   
2  04/01/18 16:56:08:568  209715392          000003.log   

                                            FullPath  \
0  \Users\jcloudy\AppData\Local\Google\Chrome\Use...   
1  \Users\jcloudy\AppData\Local\Google\Chrome\Use...   
2  \Users\jcloudy\AppData\Local\Google\Chrome\Use...   

                                           EventInfo SourceInfo FileAttribute  \
0     Data_Added / Data_Overwritten / Data_Truncated     Normal     

In [85]:
# Cell 3: Stage 1 - Keep all Basic_Info_Changed events

print("Stage 1: Filtering Basic_Info_Changed events...")

# Keep all events containing Basic_Info_Changed
bdp_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('Basic_Info_Changed', na=False)
].copy()

print(f"Basic_Info_Changed events: {len(bdp_events):,}")


Stage 1: Filtering Basic_Info_Changed events...
Basic_Info_Changed events: 5,045


In [86]:
# Cell 4: Stage 2 - Keep FIRST File_Created event per file

print("Stage 2: Filtering first creation event per file...")

# Filter to File_Created events
creation_mask = usnjrnl['EventInfo'].str.contains('File_Created', na=False)
creation_events = usnjrnl[creation_mask].copy()

# Sort by timestamp and keep first per FileReferenceNumber
creation_events = creation_events.sort_values('TimeStamp(UTC+8)')
first_creation_per_file = creation_events.groupby('FileReferenceNumber').first().reset_index()

print(f"Total File_Created events: {len(creation_events):,}")
print(f"First creation per file: {len(first_creation_per_file):,}")
print(f"Duplicate creations dropped: {len(creation_events) - len(first_creation_per_file):,}")


Stage 2: Filtering first creation event per file...
Total File_Created events: 31,499
First creation per file: 11,291
Duplicate creations dropped: 20,208


In [87]:
# Cell 5: Stage 3 - Keep Delete/Rename at monitored paths only

print("Stage 3: Filtering delete/rename events at monitored paths...")

# Get set of paths where we have creation events (monitored paths)
monitored_paths = set(first_creation_per_file['FullPath'].dropna().unique())
print(f"Monitored paths (files with creation events): {len(monitored_paths):,}")

# Filter delete/rename events to monitored paths only
delete_rename_mask = usnjrnl['EventInfo'].str.contains('File_Deleted|File_Renamed', na=False)
delete_rename_events = usnjrnl[delete_rename_mask].copy()
delete_rename_filtered = delete_rename_events[
    delete_rename_events['FullPath'].isin(monitored_paths)
].copy()

print(f"Total delete/rename events: {len(delete_rename_events):,}")
print(f"Delete/rename at monitored paths: {len(delete_rename_filtered):,}")
print(f"Delete/rename dropped (irrelevant paths): {len(delete_rename_events) - len(delete_rename_filtered):,}")


Stage 3: Filtering delete/rename events at monitored paths...
Monitored paths (files with creation events): 5,754
Total delete/rename events: 13,695
Delete/rename at monitored paths: 414
Delete/rename dropped (irrelevant paths): 13,281


In [88]:
# Cell 6: Combine all filtered events

print("Combining filtered events...")

# Concatenate all three stages
usnjrnl_filtered = pd.concat([
    bdp_events,
    first_creation_per_file,
    delete_rename_filtered
]).drop_duplicates(subset=['USN']).copy()

print(f"\nFiltering results:")
print(f"  Original events: {len(usnjrnl):,}")
print(f"  Filtered events: {len(usnjrnl_filtered):,}")
print(f"  Reduction: {len(usnjrnl) - len(usnjrnl_filtered):,} events ({(1 - len(usnjrnl_filtered)/len(usnjrnl))*100:.1f}%)")


Combining filtered events...

Filtering results:
  Original events: 101,023
  Filtered events: 16,647
  Reduction: 84,376 events (83.5%)


In [95]:
# Cell 7: Verify FileReferenceNumber column exists

print("\nVerifying required columns...")

if 'FileReferenceNumber' not in usnjrnl_filtered.columns:
    print("WARNING: FileReferenceNumber column missing!")
    print(f"Available columns: {list(usnjrnl_filtered.columns)}")
    
    # Check for alternate column names
    if 'FileReferenceNumber_event' in usnjrnl_filtered.columns:
        print("Found FileReferenceNumber_event, renaming to FileReferenceNumber")
        usnjrnl_filtered = usnjrnl_filtered.rename(columns={'FileReferenceNumber_event': 'FileReferenceNumber'})
    elif 'MFT_RecordNumber' in usnjrnl_filtered.columns:
        print("Found MFT_RecordNumber, using as FileReferenceNumber")
        usnjrnl_filtered['FileReferenceNumber'] = usnjrnl_filtered['MFT_RecordNumber']
else:
    print("FileReferenceNumber column present")

# Verify required timestamp columns
required_cols = ['SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'TimeStamp(UTC+8)']
missing_cols = [col for col in required_cols if col not in usnjrnl_filtered.columns]
if missing_cols:
    print(f"WARNING: Missing columns: {missing_cols}")
else:
    print("All required timestamp columns present")



Verifying required columns...
FileReferenceNumber column present
All required timestamp columns present


In [90]:
# Cell 8: Save filtered UsnJrnl

print("\nSaving filtered UsnJrnl...")
usnjrnl_filtered.to_csv(USNJRNL_FILTERED_PATH, index=False)
print(f"Saved to: {USNJRNL_FILTERED_PATH}")
print(f"File size: {USNJRNL_FILTERED_PATH.stat().st_size / 1024 / 1024:.2f} MB")


Saving filtered UsnJrnl...
Saved to: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1/LoneWolf_UsnJrnl_filtered.csv
File size: 5.15 MB


# Part 2: LogFile Filtering

Strategy: Keep timestamp change events and file creation/deletion events for tunneling detection

In [91]:
# Cell 9: Load LogFile

print("\nLoading LogFile...")
logfile = pd.read_csv(LOGFILE_PATH, low_memory=False)

print(f"Total events loaded: {len(logfile):,}")
print(f"Columns: {list(logfile.columns)}")
print(f"\nEvent types distribution:")
print(logfile['Event'].value_counts().head(10))



Loading LogFile...
Total events loaded: 16,882
Columns: ['LSN', 'EventTime(UTC+8)', 'Event', 'Detail', 'File/Directory Name', 'Full Path', 'CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime', 'Redo', 'Target VCN', 'Cluster Index']

Event types distribution:
Event
Updating Modified Time                  4474
File Deletion                           2259
File Creation                           2126
Writing Content of Non-Resident File    1803
Updating MFTModified Time               1287
Renaming File                            813
Writing Content of Resident File         700
# Check Point                            578
Time Reversal Event                      573
File Creation(File System Tunneling)     439
Name: count, dtype: int64


In [92]:
# Cell 10: Filter LogFile - Keep timestamp change events

print("\nFiltering LogFile events...")

# Keep timestamp change events and file creation/deletion
logfile_filtered = logfile[
    logfile['Event'].str.contains(
        'Time Reversal Event|Updating Modified Time|Updating Creation Time|File Creation|File Deletion',
        na=False,
        case=False
    )
].copy()

print(f"\nFiltering results:")
print(f"  Original events: {len(logfile):,}")
print(f"  Filtered events: {len(logfile_filtered):,}")
print(f"  Reduction: {len(logfile) - len(logfile_filtered):,} events ({(1 - len(logfile_filtered)/len(logfile))*100:.1f}%)")

print(f"\nFiltered event types:")
print(logfile_filtered['Event'].value_counts())



Filtering LogFile events...

Filtering results:
  Original events: 16,882
  Filtered events: 9,898
  Reduction: 6,984 events (41.4%)

Filtered event types:
Event
Updating Modified Time                             4474
File Deletion                                      2259
File Creation                                      2126
Time Reversal Event                                 573
File Creation(File System Tunneling)                439
Time Reversal Event & Changing FileAttribute         14
Updating Modified Time & Changing FileAttribute      13
Name: count, dtype: int64


In [93]:
# Cell 11: Save filtered LogFile

print("\nSaving filtered LogFile...")
logfile_filtered.to_csv(LOGFILE_FILTERED_PATH, index=False)
print(f"Saved to: {LOGFILE_FILTERED_PATH}")
print(f"File size: {LOGFILE_FILTERED_PATH.stat().st_size / 1024 / 1024:.2f} MB")



Saving filtered LogFile...
Saved to: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1/LoneWolf_LogFile_filtered.csv
File size: 2.56 MB


In [94]:
# Cell 12: Summary statistics

print("\n" + "="*60)
print("PHASE 1 COMPLETE - FILTERING SUMMARY")
print("="*60)

print(f"\nUsnJrnl:")
print(f"  Input:  {len(usnjrnl):,} events")
print(f"  Output: {len(usnjrnl_filtered):,} events")
print(f"  Reduction: {(1 - len(usnjrnl_filtered)/len(usnjrnl))*100:.1f}%")

print(f"\nLogFile:")
print(f"  Input:  {len(logfile):,} events")
print(f"  Output: {len(logfile_filtered):,} events")
print(f"  Reduction: {(1 - len(logfile_filtered)/len(logfile))*100:.1f}%")

print(f"\nOutput files:")
print(f"  {USNJRNL_FILTERED_PATH}")
print(f"  {LOGFILE_FILTERED_PATH}")

print("\nNext step: Phase 2 - File-Level Feature Engineering")



PHASE 1 COMPLETE - FILTERING SUMMARY

UsnJrnl:
  Input:  101,023 events
  Output: 16,647 events
  Reduction: 83.5%

LogFile:
  Input:  16,882 events
  Output: 9,898 events
  Reduction: 41.4%

Output files:
  /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1/LoneWolf_UsnJrnl_filtered.csv
  /Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1/LoneWolf_LogFile_filtered.csv

Next step: Phase 2 - File-Level Feature Engineering


In [96]:
import pandas as pd

# Load dataset
path = "/Users/soni/Github/Digital-Detectives_Thesis/notebooks/LW Testing Pipeline/Outputs/trial 2/v1/LoneWolf_UsnJrnl_filtered.csv"
df = pd.read_csv(path)

# Treat empty strings as nulls (important for formatted timestamps)
df.replace("", pd.NA, inplace=True)

# Check null values
print("Null count per column:")
print(df.isnull().sum(), "\n")

print("Percentage of nulls per column:")
print((df.isnull().mean() * 100).round(2), "\n")

print("Any nulls in dataset?:", df.isnull().values.any(), "\n")

print("Rows containing at least one null value:")
print(df[df.isnull().any(axis=1)])


Null count per column:
TimeStamp(UTC+8)                   0
USN                                0
File/Directory Name                0
FullPath                        5675
EventInfo                          0
SourceInfo                         0
FileAttribute                      0
FileReferenceNumber                0
FileName_MFT                       0
SI_CreationTime_Formatted          2
SI_ModifiedTime_Formatted          2
SI_AccessedTime_Formatted          2
SI_MFTModifiedTime_Formatted       2
EventTime_Formatted                0
MFT_RecordNumber                   0
dtype: int64 

Percentage of nulls per column:
TimeStamp(UTC+8)                 0.00
USN                              0.00
File/Directory Name              0.00
FullPath                        34.09
EventInfo                        0.00
SourceInfo                       0.00
FileAttribute                    0.00
FileReferenceNumber              0.00
FileName_MFT                     0.00
SI_CreationTime_Formatted        